# PlanetCare — IRIS GraphRAG System Demo
## iris-vector-graph + Knowledge Graph + Agents

This notebook demonstrates the full iris-vector-graph API on the PlanetCare demo dataset.
The clustering story (HDBSCAN → KB articles) is in `planetcare_clustering_demo.ipynb`.

| API | What it does |
|-----|-------------|
| `engine.kg_KNN_VEC()` | K-nearest neighbors by vector similarity |
| `engine.kg_VECTOR_GRAPH_SEARCH()` | Hybrid vector + graph traversal |
| `ops.kg_GRAPH_WALK()` | BFS/DFS from any node |
| `ops.kg_NEIGHBORHOOD_EXPANSION()` | All neighbors of a node set |
| `engine.ppr()` | Personalized PageRank |
| `engine.execute_cypher()` | Direct Cypher query |

*Requires: IRIS running (see README). Embedding provider: local MiniLM by default (no API key).*

## 0. Setup

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'setup'))

import iris
from iris_vector_graph import IRISGraphEngine
from iris_vector_graph.operators import IRISGraphOperators
from embedder import get_embedder, get_llm_client

IRIS_HOST = os.getenv('IRIS_HOST', 'localhost')
IRIS_PORT = int(os.getenv('IRIS_PORT', '11983'))

conn = iris.connect(IRIS_HOST, IRIS_PORT, 'USER', '_SYSTEM', 'SYS')
engine = IRISGraphEngine(conn, embedding_dimension=384)
ops = IRISGraphOperators(conn)
cur = conn.cursor()

embedder = get_embedder()
llm_client, llm_model = get_llm_client()

stats = engine.graph_stats()
cur.execute("SELECT COUNT(*) FROM PC.Tickets")
tickets = cur.fetchone()[0]

print(f"Connected: {IRIS_HOST}:{IRIS_PORT}")
print(f"  Embedder:    {embedder.name} ({embedder.dim}-dim)")
print(f"  PC.Tickets:  {tickets}")
print(f"  Graph nodes: {stats.get('node_count', 0):,}")
print(f"  Graph edges: {stats.get('edge_count', 0):,}")
print(f"  Embeddings:  {stats.get('embedding_count', 0):,}")
print(f"  LLM:         {llm_model or 'not configured'}")

---
## 1. Graph Stats

`engine.graph_stats()` — counts, top labels, top predicates, embedding coverage.

In [ ]:
stats = engine.graph_stats(top_n=10)

print(f"Nodes: {stats.get('node_count', 0):,}  Edges: {stats.get('edge_count', 0):,}")
print()
print("Labels:")
for label, count in list(stats.get('top_labels', {}).items())[:8]:
    print(f"  {label:<22} {count:>5}  " + "█" * min(20, count // 3))
print()
print("Predicates:")
for pred, count in list(stats.get('top_predicates', {}).items())[:6]:
    print(f"  {pred:<22} {count:>5}")

---
## 2. Vector Search — `engine.kg_KNN_VEC()`

Embed a query and find K nearest nodes. The query_vector must be passed as a JSON string.

In [ ]:
query = "billing invoice discount amount mismatch"
print(f'Query: "{query}"')

query_vec = embedder.embed([query])[0]
query_json = json.dumps(query_vec)

results = engine.kg_KNN_VEC(query_vector=query_json, k=8)

print(f"\nTop {len(results)} results:")
for node_id, score in results:
    tid = node_id.replace('pc_ticket:', '')
    cur.execute("SELECT classification, summary FROM PC.Tickets WHERE ticket_id = ?", [tid])
    r = cur.fetchone()
    cat = (r[0] or "?") if r else "?"
    summ = (r[1] or "")[:60] if r else ""
    print(f"  {node_id[-18:]:<20} {score:.4f}  [{cat}]")
    print(f"  {'':20}          {summ}")

### Hybrid Search — `engine.kg_VECTOR_GRAPH_SEARCH()`

Fuses vector similarity with graph expansion. Tickets reachable only via the graph are flagged `in_graph_expansion=True`.

In [ ]:
hybrid = engine.kg_VECTOR_GRAPH_SEARCH(query_vector=query_json, k=10, expansion_depth=1)

vec_hits = [r for r in hybrid if not r.get('in_graph_expansion')]
graph_hits = [r for r in hybrid if r.get('in_graph_expansion')]

print(f"Hybrid results: {len(hybrid)} total")
print(f"  Vector only:       {len(vec_hits)}")
print(f"  Graph-expanded:    {len(graph_hits)}")
print()
for r in hybrid[:6]:
    source = "vec  " if not r.get('in_graph_expansion') else "graph"
    nid = r['entity_id'].replace('pc_ticket:', '')
    print(f"  [{source}] {nid:<15} {r['combined_score']:.4f}")

---
## 3. Graph Walk — `ops.kg_GRAPH_WALK()`

Traverse AFFECTS / EXHIBITS / HAS_SYMPTOM / FIXED_BY edges from a ticket node.

In [ ]:
top_node = results[0][0]
tid = top_node.replace('pc_ticket:', '')

cur.execute("SELECT classification, summary FROM PC.Tickets WHERE ticket_id = ?", [tid])
r = cur.fetchone()
print(f"Starting node: {top_node}")
if r:
    print(f"  [{r[0]}] {r[1][:80]}")
print()

walk = ops.kg_GRAPH_WALK(top_node, max_depth=2, traversal_mode='BFS')

ICONS = {'AFFECTS': '🔴', 'EXHIBITS': '⚡', 'HAS_SYMPTOM': '🩺',
         'FIXED_BY': '✅', 'HAS_ROOT_CAUSE': '🔍', 'ANALYZED': '📋'}

print(f"Graph walk: {len(walk)} edges")
for src, pred, tgt, depth, path in walk[:10]:
    icon = ICONS.get(pred, '→')
    indent = '  ' * depth
    print(f"{indent}{icon} {pred:<16} {tgt[-45:]}")

---
## 4. Neighborhood Expansion — `ops.kg_NEIGHBORHOOD_EXPANSION()`

Given entity nodes (modules, errors, symptoms), find all tickets connected to them.
This is how the discovery loop works: vector search → entities → expand to new tickets.

In [ ]:
entity_nodes = [tgt for src, pred, tgt, depth, path in walk
                if pred in ('AFFECTS', 'EXHIBITS') and depth == 1]

print(f"Entity nodes from top ticket: {entity_nodes[:3]}")
print()

if entity_nodes:
    expansion = ops.kg_NEIGHBORHOOD_EXPANSION(entity_nodes[:3], expansion_depth=1)

    known = {r[0] for r in results}
    new_tickets = [(src, pred, tgt, conf) for src, pred, tgt, conf in expansion
                   if tgt not in known and 'pc_ticket' in tgt]

    print(f"Vector search found:    {len(results)} tickets")
    print(f"Graph expansion adds:   {len(set(n[2] for n in new_tickets))} more")
    print()
    seen = set()
    for src, pred, tgt, conf in new_tickets[:6]:
        if tgt not in seen:
            seen.add(tgt)
            tid2 = tgt.replace('pc_ticket:', '')
            cur.execute("SELECT classification, summary FROM PC.Tickets WHERE ticket_id = ?", [tid2])
            r2 = cur.fetchone()
            print(f"  {tgt[-18:]} [{r2[0] if r2 else '?'}]")
            print(f"    {(r2[1] or '')[:65] if r2 else ''}")
else:
    print("No entity nodes found for this ticket — try a different query")

---
## 5. Cypher Queries — `engine.execute_cypher()`

Direct graph queries using openCypher syntax. Always use `read_only=True` to prevent accidental writes.

In [ ]:
result = engine.execute_cypher(
    "MATCH (t)-[e]->(m) "
    "WHERE t.ticket_id STARTS WITH 'PC-' AND e.predicate = 'AFFECTS' "
    "RETURN m.node_id AS module, COUNT(DISTINCT t.node_id) AS ticket_count "
    "ORDER BY ticket_count DESC LIMIT 8",
    read_only=True
)

print("Most affected modules across all PlanetCare tickets:")
rows = result.get('rows', [])
if rows:
    for row in rows:
        module = str(row[0] if row else '?').replace('PC_MODULE:', '')
        count = row[1] if len(row) > 1 else '?'
        bar = '█' * min(20, int(count) // 2 if isinstance(count, int) else 5)
        print(f"  {module:<30} {count:>4}  {bar}")
else:
    entity_result = engine.execute_cypher(
        "MATCH (n) WHERE n.node_id STARTS WITH 'PC_MODULE:' "
        "RETURN n.node_id, n.text LIMIT 8",
        read_only=True
    )
    print("PlanetCare modules in graph:")
    for row in entity_result.get('rows', [])[:8]:
        print(f"  {row[0]}: {row[1] if len(row) > 1 else ''}")

---
## 6. KB Article Synthesis

Collect resolved tickets from a category cluster and synthesize a KB article.

In [ ]:
if llm_client:
    cur.execute(
        "SELECT ticket_id, summary, description, resolution "
        "FROM PC.Tickets WHERE classification = 'BILLING' AND status = 'Closed' "
        "ORDER BY ticket_id"
    )
    resolved = cur.fetchall()[:4]
    context = "\n---\n".join(
        f"{tid}: {s}\n{('Resolution: ' + str(r)[:200]) if r else ''}"
        for tid, s, d, r in resolved
    )
    resp = llm_client.chat.completions.create(
        model=llm_model,
        messages=[
            {"role": "system", "content": "PlanetCare EMR KB writer. Write: Title, Problem, Root Cause, Resolution Steps, Prevention. Max 200 words."},
            {"role": "user", "content": f"KB article from PlanetCare billing tickets:\n\n{context}"}
        ],
        max_tokens=280
    )
    print("GENERATED KB ARTICLE")
    print("="*55)
    print(resp.choices[0].message.content)
    print(f"\nSource: {len(resolved)} resolved billing tickets")
else:
    print("LLM not configured — set OPENAI_API_KEY or OPENROUTER_API_KEY")
    print("Showing resolved billing tickets instead:")
    cur.execute("SELECT ticket_id, summary FROM PC.Tickets WHERE classification='BILLING' AND status='Closed'")
    for tid, summ in cur.fetchall()[:5]:
        print(f"  {tid}: {summ[:70]}")

---
## Summary

| Engine call | Result |
|-------------|--------|
| `kg_KNN_VEC(query_json, k=8)` | Top-8 billing tickets by similarity |
| `kg_VECTOR_GRAPH_SEARCH(query_json)` | Hybrid: vector + graph-expanded tickets |
| `kg_GRAPH_WALK(node, depth=2)` | Entity relationships (AFFECTS/EXHIBITS/...) |
| `kg_NEIGHBORHOOD_EXPANSION(entities)` | Adjacent tickets not in vector results |
| `ppr(seed, top_k=10)` | Graph-proximity ranking |
| `execute_cypher(query)` | Direct openCypher queries |

**All queries run on local IRIS — no external database needed.**

See `planetcare_clustering_demo.ipynb` for the HDBSCAN clustering workflow that generated the most discussion at READY 2026.